# Retriever
- 비정형 질의(query)를 입력 받아 Vector store에서 관련된 내용을 검색하는 기능을 제공하는 인터페이스로 다양한 데이터 소스에서 정보를 검색하여 대규모 언어 모델(LLM) 기반 애플리케이션의 **정확성을** 향상시키는 데 핵심적인 역할을 한다.

![RAG](figures/rag2.png)

## 주요 특징
- **다양한 데이터 소스 지원**
	- Retriever는 벡터 스토어, 그래프 데이터베이스, 관계형 데이터베이스 등 여러 종류의 검색 시스템과 상호작용할 수 있는 통일된 인터페이스를 제공한다다.
- **간단한 인터페이스**: Retriever는 문자열 형태의 쿼리를 입력받아 관련 문서의 리스트를 반환한다. 이러한 단순한 구조 덕분에 다양한 검색 시스템과 쉽게 통합할 수 있다. 

### VectorStore와 Retriever

VectorStore와 Retriever는 모두 Vector Database 검색(Retrieval)과 관련되지만, 역할과 책임이 다르다.
- **VectorStore**
  - VectorStore는 벡터 데이터를 관리하는 저장소로 다음과 같은 기능을 직접 제공한다.

  	- 저장: 문서를 벡터(임베딩)로 변환해 데이터베이스에 저장
  	- 삭제/수정: 저장된 벡터를 지우거나 업데이트
  	- 인덱싱: 빠르게 검색할 수 있도록 벡터 인덱스를 생성·관리
  	- 유사도 검색: 입력 질문과 가장 비슷한 벡터를 찾아 관련 문서를 반환
  - 즉, VectorStore는 **벡터 데이터 관리 + 벡터 검색 기능을 모두 포함한 인터페이스**이다.
  - 내부적으로는 실제 벡터 DB(예: FAISS, Chroma, Pinecone, Milvus 등)와 연결되어 있고, 그 DB를 직접 다루는 역할을 한다.

- **Retriever**
	- Retriever는 "검색 요청을 어떻게 할지"만 정리해 둔 실행 모듈로 직접 벡터를 저장하거나 인덱스를 만들지는 않는다.
	- 검색 개수(k), 메타데이터 필터, 하이브리드 검색 여부, 재랭킹 적용 여부 등 설정된 전략에 따라 VectorStore(또는 다른 검색 백엔드)에 검색을 요청하고 결과만 받아서 반환한다.
	- Runnable 타입이므로 체인(LCEL) 안에 포함될 수 있다.

## 다양한 Retriever 방식

- Retriever의 검색 성능을 높이기 위한 다양한 Retriever의 확장 전략이 있다. 

1. **벡터 스토어(Vector Store) Retriever**
   - VectorStore로 부터 유사도를 기반으로 검색하는 **가장 기본 Retriever**
   - 텍스트 조각(청크)마다 **임베딩(embedding)을** 생성하여 벡터 공간에 저장하고, 쿼리 임베딩과의 **코사인 유사도(cosine similarity)** 등을 기반으로 유사한 텍스트를 검색한다.
   - 검색 속도가 빠르고 구현이 간단하여, 기본적인 검색 시스템을 구축할 때 적합하다.
2. **[ParentDocumentRetriever](https://python.langchain.com/docs/how_to/parent_document_retriever/)**
   - 하나의 문서를 여러 청크로 나눈 뒤 각각을 인덱싱하고, 쿼리와 가장 유사한 청크를 찾은 다음 해당 청크가 속한 **전체 원본 문서**를 반환한다.
   - 작은 정보 조각들이 모여 하나의 문서를 구성할 때 유용하며, 문맥을 넓게 유지할 수 있다.
3. **[MultiVectorRetriever](https://python.langchain.com/docs/how_to/multi_vector/)**
   - 각 문서에 대해 요약을 하거나, 가상의 질문을 생성하거나, 사람이 중요한 내용을 직접 추가하여 문서당 여러 개의 임베딩 벡터를 생성한다.
   - 텍스트 전체보다 더 핵심적인 정보가 검색에 반영되도록 하고자 할 때 효과적이다.
   - 특히, 문서가 길거나, 중요한 내용이 문서의 특정 부분에 집중되어 있는 경우에 유리하다.
4. **[SelfQueryRetriever](https://python.langchain.com/docs/how_to/self_query/)**
   - 대규모 언어 모델(LLM, Large Language Model)을 활용하여 사용자의 질문을 적절한 검색어와 **메타데이터(metadata)** 필터로 자동 변환한다.
   - 예를 들어, 문서의 작성자, 날짜, 태그와 같은 메타데이터를 기준으로 검색할 수 있다.
   - 문서 자체의 내용뿐만 아니라, 문서에 부가된 속성 정보에 대해 질문할 때 유용하다.
5. **[ContextualCompressionRetriever](https://python.langchain.com/docs/how_to/contextual_compression/)**
   - 기존 Retriever와 조합되어 사용된다.
   - 먼저 일반적인 검색을 수행한 후, 검색된 문서들에서 쿼리와 관련 없는 불필요한 정보를 제거하고 핵심 내용만을 추출하여 반환한다.
   - 정보를 요약하거나 압축하여 LLM에 전달할 문서 길이를 줄일 때 유용하다.
6. **[MultiQueryRetriever](https://python.langchain.com/docs/how_to/MultiQueryRetriever/)**
   - LLM을 이용해 하나의 쿼리로부터 여러 가지 변형된 쿼리를 생성하고, 각 쿼리에 대해 검색을 수행한 뒤 결과를 합치는 방식.
   - 다양한 표현에 강해 검색 범위를 넓히고 성능을 높인다.
7. **[EnsembleRetriever](https://python.langchain.com/docs/how_to/ensemble_retriever/)**
   - 여러 개의 Retriever(예: BM25, 벡터 기반 등)를 결합해 가중치를 기반으로 결과를 조합(re-ranking)한다.
   - 서로 다른 장점을 가진 Retriever를 하나로 묶어 성능을 강화한다.